# DeBERTa-v3-large Reasoning Scorer for Cryptic Wordplay — SLURM-safe

This version is adapted for the TAU SLURM environment and includes numerical-stability checks.

**Important:** it starts with `TEST_MODE = True`, which runs only 10 training steps. After a successful test with zero NaN/Inf tensors, change it to `False` for the full 3-epoch run.


## 1. Install dependencies

In Colab, first select **Runtime → Change runtime type → GPU**.

In [ ]:

# On the TAU SLURM cluster, use the already-created venv instead of upgrading packages
# from inside the notebook. In Google Colab only, install the dependencies.

import os
import sys
import subprocess

IN_COLAB = os.path.exists("/content") and "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers", "datasets", "accelerate", "sentencepiece", "scikit-learn"
    ])
    print("Installed/updated dependencies for Colab.")
else:
    print("SLURM/local environment detected: using the existing Python environment.")


## 2. Imports and configuration

In [ ]:

import os
import re
import random
import inspect
import shutil

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset, Dataset
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

SEED = 42
MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LENGTH = 256

NUM_EPOCHS = 3
LEARNING_RATE = 1e-6
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0

# Safety mode: first run only 10 optimizer steps.
# After it finishes with "Bad tensors after training: 0", change to False.
TEST_MODE = False
TEST_MAX_STEPS = 10

OUTPUT_ROOT = os.environ.get(
    "OUTPUT_ROOT",
    os.path.join("outputs", "notebook_deberta_fixed"),
)

os.makedirs(OUTPUT_ROOT, exist_ok=True)

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)
print("TEST_MODE:", TEST_MODE)
print("Output root:", OUTPUT_ROOT)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 3. Load the Wordplay dataset from GitHub

The public sample in `mdda/cryptic-wordplay` contains `train` and `val` JSONL files.
Each item includes fields such as `clue`, `answer`, `pattern`, and `wordplay`.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/mdda/cryptic-wordplay/main/prebuilt"

data_files = {
    "train": f"{BASE_URL}/sample_teacow_train.jsonl",
    "validation": f"{BASE_URL}/sample_teacow_val.jsonl",
}

raw_dataset = load_dataset("json", data_files=data_files)

print(raw_dataset)
print("\nExample:")
print(raw_dataset["train"][0])

## 4. Basic cleaning and sanity checks

In [ ]:
def is_usable(example):
    required = ["clue", "answer", "wordplay", "pattern"]
    for key in required:
        value = example.get(key)
        if value is None:
            return False
        if not str(value).strip():
            return False
    return True


raw_dataset = raw_dataset.filter(is_usable)

print("Train examples:", len(raw_dataset["train"]))
print("Validation examples:", len(raw_dataset["validation"]))

print("\nColumns:")
print(raw_dataset["train"].column_names)

In [ ]:
def clean_clue(clue):
    # The dataset marks the definition span with {...}.
    # Remove the braces so the scorer cannot exploit an annotation
    # that would normally be unavailable at inference time.
    return str(clue).replace("{", "").replace("}", "")


for i in range(3):
    ex = raw_dataset["train"][i]
    print("Original:", ex["clue"])
    print("Clean:   ", clean_clue(ex["clue"]))
    print("Answer:  ", ex["answer"])
    print("Wordplay:", ex["wordplay"])
    print("-" * 100)

## 5. Build positive and negative reasoning examples

Each original item produces one pair:

- label `1`: correct reasoning
- label `0`: incorrect reasoning

The wrong reasoning is preferably taken from another clue with the same `pattern`.
This makes the negative less trivial than completely random text.

`pair_id` is kept only for later ranking evaluation.

In [ ]:
def create_scoring_dataset(split, seed=42, shuffle=True):
    rng = random.Random(seed)
    examples = list(split)

    by_pattern = {}
    for i, ex in enumerate(examples):
        pattern = str(ex["pattern"])
        by_pattern.setdefault(pattern, []).append(i)

    rows = []

    for i, ex in enumerate(examples):
        clue = clean_clue(ex["clue"])
        answer = str(ex["answer"]).strip()
        correct_reasoning = str(ex["wordplay"]).strip()
        pattern = str(ex["pattern"])

        rows.append({
            "pair_id": i,
            "clue": clue,
            "answer": answer,
            "reasoning": correct_reasoning,
            "labels": 1,
        })

        same_pattern_candidates = [
            j for j in by_pattern.get(pattern, [])
            if (
                j != i
                and str(examples[j]["wordplay"]).strip() != correct_reasoning
                and str(examples[j]["answer"]).strip() != answer
            )
        ]

        if same_pattern_candidates:
            negative_idx = rng.choice(same_pattern_candidates)
        else:
            fallback = [
                j for j in range(len(examples))
                if (
                    j != i
                    and str(examples[j]["wordplay"]).strip() != correct_reasoning
                )
            ]
            negative_idx = rng.choice(fallback)

        negative_example = examples[negative_idx]

        wrong_answer = str(negative_example["answer"]).strip()
        wrong_reasoning = str(negative_example["wordplay"]).strip()

        rows.append({
            "pair_id": i,
            "clue": clue,
            "answer": wrong_answer,
            "reasoning": wrong_reasoning,
            "labels": 0,
        })

    if shuffle:
        rng.shuffle(rows)

    return Dataset.from_list(rows)


train_scoring = create_scoring_dataset(
    raw_dataset["train"],
    seed=SEED,
    shuffle=True,
)

val_scoring = create_scoring_dataset(
    raw_dataset["validation"],
    seed=SEED + 1,
    shuffle=True,
)

print(train_scoring)
print(val_scoring)

train_labels = np.array(train_scoring["labels"])
val_labels = np.array(val_scoring["labels"])

print("\nTrain positive fraction:", train_labels.mean())
print("Validation positive fraction:", val_labels.mean())

In [ ]:
positive = next(x for x in train_scoring if x["labels"] == 1)
negative = next(x for x in train_scoring if x["labels"] == 0)

for name, ex in [("POSITIVE", positive), ("NEGATIVE", negative)]:
    print("\n" + "=" * 100)
    print(name)
    print("Clue:     ", ex["clue"])
    print("Answer:   ", ex["answer"])
    print("Reasoning:", ex["reasoning"])
    print("Label:    ", ex["labels"])

## 6. Format the model input

We explicitly identify each component:

```text
CLUE: ...
ANSWER: ...
REASONING: ...
```

DeBERTa then performs binary sequence classification over the entire input.

In [ ]:
def format_input(example):
    return (
        f"CLUE: {example['clue']}\n"
        f"ANSWER: {example['answer']}\n"
        f"REASONING: {example['reasoning']}"
    )


print(format_input(train_scoring[0]))

## 7. Load DeBERTa from Hugging Face

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    dtype=torch.float32,  # Force master weights to FP32
    id2label={
        0: "INCORRECT",
        1: "CORRECT",
    },
    label2id={
        "INCORRECT": 0,
        "CORRECT": 1,
    },
)

model.to(device)

model_dtype = next(model.parameters()).dtype

bad_before = [
    name
    for name, p in model.named_parameters()
    if not torch.isfinite(p).all()
]

print("Loaded:", MODEL_NAME)
print("Number of labels:", model.config.num_labels)
print("Model dtype:", model_dtype)
print("Bad tensors immediately after loading:", len(bad_before))

assert model_dtype == torch.float32, f"Expected float32 weights, got {model_dtype}"
assert len(bad_before) == 0, "The pretrained model already contains NaN/Inf before training."


You may see a warning that the classification head was newly initialized.
That is expected: the pretrained DeBERTa encoder is loaded from Hugging Face,
while the final binary classification layer must be learned on this task.

## 8. Tokenize the datasets

In [ ]:
def tokenize_function(example):
    text = format_input(example)

    return tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
    )


tokenized_train = train_scoring.map(
    tokenize_function,
    remove_columns=["pair_id", "clue", "answer", "reasoning"],
)

tokenized_val = val_scoring.map(
    tokenize_function,
    remove_columns=["pair_id", "clue", "answer", "reasoning"],
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_train)
print("\nTokenized example:")
print(tokenized_train[0])

## 9. Metrics

In [ ]:
def softmax_np(logits):
    shifted = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if isinstance(logits, tuple):
        logits = logits[0]

    probs = softmax_np(logits)
    predictions = np.argmax(probs, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        pos_label=1,
        zero_division=0,
    )

    try:
        auc = roc_auc_score(labels, probs[:, 1])
    except ValueError:
        auc = float("nan")

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc,
    }

## 10. Training arguments

The code below handles both newer Transformers versions that use
`eval_strategy` and older versions that use `evaluation_strategy`.

In [ ]:

training_kwargs = dict(
    output_dir=os.path.join(OUTPUT_ROOT, "checkpoints"),
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    max_grad_norm=MAX_GRAD_NORM,
    weight_decay=WEIGHT_DECAY,
    logging_strategy="steps",
    logging_steps=1 if TEST_MODE else 50,
    logging_nan_inf_filter=False,
    fp16=torch.cuda.is_available(),  # AMP + GradScaler; master weights remain FP32
    bf16=False,
    report_to="none",
    seed=SEED,
)

signature = inspect.signature(TrainingArguments.__init__)

if TEST_MODE:
    # Short numerical-stability test: no checkpoint/eval scheduling needed.
    training_kwargs["max_steps"] = TEST_MAX_STEPS
    training_kwargs["save_strategy"] = "no"
    training_kwargs["load_best_model_at_end"] = False

    if "eval_strategy" in signature.parameters:
        training_kwargs["eval_strategy"] = "no"
    else:
        training_kwargs["evaluation_strategy"] = "no"
else:
    training_kwargs["save_strategy"] = "epoch"
    training_kwargs["load_best_model_at_end"] = True
    training_kwargs["metric_for_best_model"] = "auc"
    training_kwargs["greater_is_better"] = True
    training_kwargs["save_total_limit"] = 2

    if "eval_strategy" in signature.parameters:
        training_kwargs["eval_strategy"] = "epoch"
    else:
        training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

print("Training arguments ready.")
print("fp16:", training_args.fp16)
print("train batch size:", TRAIN_BATCH_SIZE)
print("gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
if TEST_MODE:
    print("TEST ONLY - max steps:", TEST_MAX_STEPS)


## 11. Create the Hugging Face Trainer

In [ ]:

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer_signature = inspect.signature(Trainer.__init__)

if "processing_class" in trainer_signature.parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer ready.")


## 12. Fine-tune DeBERTa

In [ ]:

print("Model dtype before training:", next(model.parameters()).dtype)

bad_before_training = [
    name
    for name, p in model.named_parameters()
    if not torch.isfinite(p).all()
]
print("Bad tensors before training:", len(bad_before_training))
assert len(bad_before_training) == 0, "Model contains NaN/Inf before trainer.train()."

# One real batch forward pass before training
preflight_examples = [tokenized_train[i] for i in range(min(2, len(tokenized_train)))]
preflight_batch = data_collator(preflight_examples)
preflight_batch = {k: v.to(device) for k, v in preflight_batch.items()}

model.eval()
with torch.no_grad():
    preflight_logits = model(**preflight_batch).logits

print("Pre-training logits finite:", bool(torch.isfinite(preflight_logits).all()))
print("Pre-training logits:", preflight_logits.detach().float().cpu().numpy())

assert torch.isfinite(preflight_logits).all(), "Forward pass is NaN/Inf BEFORE training."

model.train()

print(f"TrainingArguments fp16 setting: {training_args.fp16}")
train_result = trainer.train()

bad_after_training = [
    name
    for name, p in model.named_parameters()
    if not torch.isfinite(p).all()
]

print("Bad tensors after training:", len(bad_after_training))

if bad_after_training:
    print("First bad tensors:", bad_after_training[:20])
    raise RuntimeError(
        "Training produced NaN/Inf parameters. Model will NOT be saved."
    )

SAVE_DIR = os.path.join(
    OUTPUT_ROOT,
    "test_model" if TEST_MODE else "final_model",
)

os.makedirs(SAVE_DIR, exist_ok=True)
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Model safely saved to:", SAVE_DIR)
print(train_result)


## 13. Validation metrics

In [ ]:

val_metrics = trainer.evaluate()

print("Validation results")
print("-" * 60)

if TEST_MODE:
    print("NOTE: TEST_MODE=True, so these metrics are only a 10-step sanity check.")

for key, value in val_metrics.items():
    if isinstance(value, float):
        print(f"{key:25s}: {value:.4f}")
    else:
        print(f"{key:25s}: {value}")


## 14. Plot train / validation loss

This is useful for spotting overfitting.

In [ ]:
history = pd.DataFrame(trainer.state.log_history)

plt.figure(figsize=(8, 5))

if "loss" in history.columns:
    train_logs = history[history["loss"].notna()]
    if len(train_logs):
        plt.plot(train_logs["step"], train_logs["loss"], label="train loss")

if "eval_loss" in history.columns:
    eval_logs = history[history["eval_loss"].notna()]
    if len(eval_logs):
        plt.plot(
            eval_logs["step"],
            eval_logs["eval_loss"],
            marker="o",
            label="validation loss",
        )

plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("DeBERTa reasoning scorer")
plt.legend()
plt.show()

## 15. Single reasoning score

This is the main function you will use after training.

The returned value is:

\[
P(\text{CORRECT}\mid \text{clue},\text{answer},\text{reasoning})
\]

In [ ]:
def reasoning_score(clue, answer, reasoning):
    model.eval()

    example = {
        "clue": clean_clue(clue),
        "answer": str(answer).strip(),
        "reasoning": str(reasoning).strip(),
    }

    text = format_input(example)

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1)

    return probabilities[0, 1].item()

### Try the scorer on a real validation example

In [ ]:
example = raw_dataset["validation"][0]
wrong_example = raw_dataset["validation"][1]

clue = clean_clue(example["clue"])
answer = example["answer"]

correct_reasoning = example["wordplay"]
wrong_reasoning = wrong_example["wordplay"]

correct_score = reasoning_score(clue, answer, correct_reasoning)
wrong_score = reasoning_score(clue, answer, wrong_reasoning)

print("CLUE:  ", clue)
print("ANSWER:", answer)

print("\nCorrect reasoning:")
print(correct_reasoning)
print(f"Score: {correct_score:.4f}")

print("\nWrong reasoning:")
print(wrong_reasoning)
print(f"Score: {wrong_score:.4f}")

## 16. Score several candidate reasonings efficiently

Use this when another model generates several candidate chains of reasoning
and you want DeBERTa to rank them.

In [ ]:

def reasoning_scores_batch(clues, answers, reasonings, batch_size=32):
    assert len(clues) == len(answers) == len(reasonings)

    model.eval()
    all_scores = []

    texts = [
        format_input({
            "clue": clean_clue(clue),
            "answer": str(answer).strip(),
            "reasoning": str(reasoning).strip(),
        })
        for clue, answer, reasoning in zip(clues, answers, reasonings)
    ]

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            logits = model(**inputs).logits

            if not torch.isfinite(logits).all():
                raise RuntimeError(
                    f"Non-finite logits detected in scoring batch starting at index {start}."
                )

            probs = torch.softmax(logits.float(), dim=-1)[:, 1]

        all_scores.extend(probs.cpu().tolist())

    scores = np.asarray(all_scores, dtype=np.float64)

    if not np.isfinite(scores).all():
        raise RuntimeError("Non-finite reasoning scores detected.")

    return scores


def rank_reasonings(clue, answer, candidate_reasonings):
    scores = reasoning_scores_batch(
        clues=[clue] * len(candidate_reasonings),
        answers=[answer] * len(candidate_reasonings),
        reasonings=candidate_reasonings,
    )

    ranked = sorted(
        zip(candidate_reasonings, scores),
        key=lambda x: x[1],
        reverse=True,
    )

    return ranked


In [ ]:
ex = raw_dataset["validation"][0]

candidate_reasonings = [
    ex["wordplay"],
    raw_dataset["validation"][1]["wordplay"],
    raw_dataset["validation"][2]["wordplay"],
]

ranking = rank_reasonings(
    clue=clean_clue(ex["clue"]),
    answer=ex["answer"],
    candidate_reasonings=candidate_reasonings,
)

print("Clue:", clean_clue(ex["clue"]))
print("Answer:", ex["answer"])
print()

for rank, (reasoning, score) in enumerate(ranking, start=1):
    print(f"{rank}. score={score:.4f}")
    print(reasoning)
    print("-" * 80)

## 17. Pairwise ranking accuracy

Classification accuracy asks whether each item crosses a fixed decision threshold.

For a scorer, a more direct question is:

> Given the same clue and answer, does DeBERTa assign a higher score to the
> correct reasoning than to the negative reasoning?

The metric below measures exactly that.

In [ ]:

val_df = val_scoring.to_pandas()

val_df["score"] = reasoning_scores_batch(
    clues=val_df["clue"].tolist(),
    answers=val_df["answer"].tolist(),
    reasonings=val_df["reasoning"].tolist(),
    batch_size=EVAL_BATCH_SIZE,
)

VAL_SCORES_PATH = os.path.join(
    OUTPUT_ROOT,
    "validation_scores_test.csv" if TEST_MODE else "validation_scores.csv",
)
val_df.to_csv(VAL_SCORES_PATH, index=False)

print("Saved raw validation scores to:", VAL_SCORES_PATH)
print("\nLabel counts:")
print(val_df["labels"].value_counts(dropna=False))
print("\nNaN scores:", int(val_df["score"].isna().sum()), "/", len(val_df))
print("Finite scores:", int(np.isfinite(val_df["score"]).sum()), "/", len(val_df))

comparisons = []
margins = []
valid_pair_ids = []

for pair_id, group in val_df.groupby("pair_id"):
    pos = group[group["labels"] == 1]
    neg = group[group["labels"] == 0]

    if len(pos) != 1 or len(neg) != 1:
        continue

    pos_score = float(pos.iloc[0]["score"])
    neg_score = float(neg.iloc[0]["score"])

    if not np.isfinite(pos_score) or not np.isfinite(neg_score):
        continue

    comparisons.append(pos_score > neg_score)
    margins.append(pos_score - neg_score)
    valid_pair_ids.append(pair_id)

if not comparisons:
    raise RuntimeError("No valid positive-negative validation pairs could be evaluated.")

pairwise_accuracy = float(np.mean(comparisons))
mean_margin = float(np.mean(margins))

# Keep a pair_table for the next analysis cell
pair_table = val_df[val_df["pair_id"].isin(valid_pair_ids)].pivot_table(
    index="pair_id",
    columns="labels",
    values="score",
    aggfunc="first",
)
pair_table = pair_table.dropna(subset=[0, 1])
pair_table["margin"] = pair_table[1] - pair_table[0]

print(f"\nPairwise ranking accuracy: {pairwise_accuracy:.4f}")
print(f"Mean positive-negative score margin: {mean_margin:.4f}")
print(f"Number of evaluated pairs: {len(comparisons)}")

if TEST_MODE:
    print("\nNOTE: TEST_MODE=True. These are sanity-check metrics, not final results.")


## 18. Inspect ranking mistakes

These are especially useful for improving the negative-generation strategy.

In [ ]:

if "pair_table" not in globals() or pair_table.empty:
    print("No valid pair_table is available for mistake inspection.")
else:
    worst_pair_ids = (
        pair_table
        .sort_values("margin")
        .head(5)
        .index
        .tolist()
    )

    for pair_id in worst_pair_ids:
        pair_rows = val_df[val_df["pair_id"] == pair_id].sort_values(
            "labels",
            ascending=False,
        )

        print("=" * 110)

        for _, row in pair_rows.iterrows():
            kind = "TRUE" if row["labels"] == 1 else "NEGATIVE"

            print(f"\n{kind} | score={row['score']:.4f}")
            print("Clue:     ", row["clue"])
            print("Answer:   ", row["answer"])
            print("Reasoning:", row["reasoning"])


## 19. Save validation scores to CSV

In [ ]:

print("Validation scores are already saved at:")
print(VAL_SCORES_PATH)


## 20. Save the fine-tuned model

In [ ]:

print("The model was already saved immediately after the finite-weight check.")
print("Saved model directory:", SAVE_DIR)

print("\nFiles:")
for name in sorted(os.listdir(SAVE_DIR)):
    print(" -", name)


### Optional: zip the model for download

In [ ]:

archive_base = SAVE_DIR.rstrip("/")

archive_path = shutil.make_archive(
    archive_base,
    "zip",
    root_dir=os.path.dirname(SAVE_DIR),
    base_dir=os.path.basename(SAVE_DIR),
)

print("Created:", archive_path)


## 21. Optional: copy the trained model to Google Drive

Run this only if you want the checkpoint to persist after the Colab session ends.

In [ ]:
# from google.colab import drive
# import shutil
#
# drive.mount("/content/drive")
#
# DRIVE_SAVE_DIR = "/content/drive/MyDrive/deberta-wordplay-reasoning-scorer-final"
# shutil.copytree(SAVE_DIR, DRIVE_SAVE_DIR, dirs_exist_ok=True)
#
# print("Copied model to:", DRIVE_SAVE_DIR)

## Next step after the 10-step test

1. Confirm that the training cell prints `Bad tensors after training: 0`.
2. Confirm that all validation scores are finite.
3. Change `TEST_MODE = True` to `TEST_MODE = False` in the configuration cell.
4. Re-run the notebook as a SLURM job for the full 3 epochs.
